# SafeVision AI — Entrenamiento del modelo de caídas

Este cuaderno entrena un modelo YOLO propio que detecta personas caídas. Se ejecuta en Google Colab con GPU gratuita.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)*.

Pasos: 1) comprobar la GPU, 2) descargar el código, 3) descargar el dataset, 4) entrenar, 5) revisar resultados, 6) descargar el modelo.

## 1. Comprobar la GPU

In [ ]:
!nvidia-smi

## 2. Descargar el código del módulo IoT

In [ ]:
!git clone https://github.com/Luisaparrac/SafeVisionAI-IoT.git
%cd SafeVisionAI-IoT
!pip install -q ultralytics pyyaml roboflow

## 3. Descargar el dataset

En [Roboflow Universe](https://universe.roboflow.com/search?q=class%3Afall) elige un dataset de caídas, pulsa **Download Dataset → formato YOLOv11 → Show download code** y pega aquí las líneas que te da Roboflow (incluyen tu API key).

También puedes subir un `.zip` en formato YOLO y descomprimirlo en `datasets/`.

In [ ]:
from roboflow import Roboflow

# Reemplaza estas líneas por las que copia Roboflow:
rf = Roboflow(api_key="TU_API_KEY")
project = rf.workspace("WORKSPACE").project("PROYECTO")
dataset = project.version(1).download("yolov11")

DATA_YAML = f"{dataset.location}/data.yaml"
print(open(DATA_YAML).read())

## 4. Entrenar

Parte de `yolo11n.pt`, ya entrenado con el dataset COCO, y lo ajusta a las caídas (*transfer learning*). Con una GPU T4 y unas 3.000 imágenes, 50 épocas tardan alrededor de una hora.

In [ ]:
!python training/train_fall_model.py --data {DATA_YAML} --epochs 50 --imgsz 640 --batch 16 --device 0

## 5. Revisar los resultados

- **Precision:** de las caídas que detecta el modelo, cuántas eran caídas reales.
- **Recall:** de las caídas reales, cuántas detectó.
- **mAP50:** calidad general de la detección (de 0 a 1; mientras más alto, mejor).

In [ ]:
from IPython.display import Image, display
for name in ["results.png", "confusion_matrix.png", "val_batch0_pred.jpg"]:
    display(Image(filename=f"runs/fall_model/{name}", width=900))

## 6. Descargar el modelo

Guarda `fall_model.pt` en la carpeta `models/` del módulo IoT y agrega al `.env`:

```
FALL_MODEL=models/fall_model.pt
```

In [ ]:
from google.colab import files
files.download("models/fall_model.pt")